In [1]:
import pandas as pd
import numpy as np
import sqlite3

In [2]:
# Load raw data
df = pd.read_csv("data/BankChurners.csv")

In [3]:
# Generate synthetic marketing campaigns table
np.random.seed(42)
channels = ["Google Ads", "Facebook", "Email", "Referral", "Branch"]
campaigns = pd.DataFrame({
    "campaign_id": range(1, 51),
    "channel": np.random.choice(channels, 50),
    "month": np.random.choice(range(1, 13), 50),
    "spend_usd": np.random.randint(500, 10000, 50),
    "leads_generated": np.random.randint(50, 500, 50),
    "new_customers": np.random.randint(10, 200, 50),
})


In [4]:
# Connect to SQLite
conn = sqlite3.connect("data/credit_card.db")
df.to_sql("customers", conn, if_exists="replace", index=False)
campaigns.to_sql("campaigns", conn, if_exists="replace", index=False)

50

In [5]:
# --- KPI Queries ---

# 1. Conversion Rate by Channel
conversion_rate = pd.read_sql("""
    SELECT channel,
           SUM(new_customers) AS total_customers,
           SUM(leads_generated) AS total_leads,
           ROUND(100.0 * SUM(new_customers) / SUM(leads_generated), 2) AS conversion_rate_pct
    FROM campaigns
    GROUP BY channel
    ORDER BY conversion_rate_pct DESC
""", conn)

In [6]:
# 2. Campaign ROI
campaign_roi = pd.read_sql("""
    SELECT channel,
           SUM(spend_usd) AS total_spend,
           SUM(new_customers) AS customers_acquired,
           ROUND(SUM(spend_usd) * 1.0 / SUM(new_customers), 2) AS cost_per_acquisition
    FROM campaigns
    GROUP BY channel
""", conn)

In [7]:
# 3. Customer Lifetime Value proxy (avg credit limit × avg transaction count / 12)
clv = pd.read_sql("""
    SELECT Customer_Age,
           Card_Category,
           Income_Category,
           ROUND(AVG(Credit_Limit), 2) AS avg_credit_limit,
           ROUND(AVG(Total_Trans_Ct), 2) AS avg_transactions,
           ROUND(AVG(Credit_Limit) * AVG(Total_Trans_Ct) / 12.0, 2) AS estimated_clv
    FROM customers
    GROUP BY Card_Category, Income_Category
    ORDER BY estimated_clv DESC
""", conn)

In [8]:
# 4. Monthly acquisition trend (simulate from customer data)
acquisition_trend = pd.read_sql("""
    SELECT month, SUM(new_customers) AS new_customers, SUM(spend_usd) AS spend
    FROM campaigns
    GROUP BY month
    ORDER BY month
""", conn)

In [9]:
# 5. Attrition rate by segment
attrition = pd.read_sql("""
    SELECT Card_Category, Income_Category,
           COUNT(*) AS total,
           SUM(CASE WHEN Attrition_Flag = 'Attrited Customer' THEN 1 ELSE 0 END) AS churned,
           ROUND(100.0 * SUM(CASE WHEN Attrition_Flag = 'Attrited Customer' THEN 1 ELSE 0 END) / COUNT(*), 2) AS churn_rate_pct
    FROM customers
    GROUP BY Card_Category, Income_Category
""", conn)

In [10]:
# Save all to Excel for Power BI import
with pd.ExcelWriter("data/analytics_output.xlsx") as writer:
    conversion_rate.to_excel(writer, sheet_name="Conversion_Rate", index=False)
    campaign_roi.to_excel(writer, sheet_name="Campaign_ROI", index=False)
    clv.to_excel(writer, sheet_name="CLV", index=False)
    acquisition_trend.to_excel(writer, sheet_name="Acquisition_Trend", index=False)
    attrition.to_excel(writer, sheet_name="Attrition", index=False)

print("SQL pipeline complete. analytics_output.xlsx saved.")
conn.close()

SQL pipeline complete. analytics_output.xlsx saved.
